In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

In [5]:
data_path = "../Dataset/SMT_Dataset/preprocessed_human_smt_dataset.csv"
df = pd.read_csv(data_path)

In [7]:
df['movement_speed'] = df['movement_dist'] / df['completion_time']
df['accuracy'] = 1/df['rmsd']
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()
df.columns

Index(['participant_id', 'session_no', 'task_type', 'trial_no', 'day', 'block',
       'start_point_x', 'start_point_y', 'target_point_x', 'target_point_y',
       'start_time', 'end_time', 'quadrant', 'is_success', 'actual_dist',
       'movement_dist', 'completion_time', 'path', 'time_string',
       'time_diff_ms', 'Age', 'Cohort', 'Gestational_Age',
       'mabc_total_test_score', 'mabc_standard_score', 'mabc_percentile',
       'distances', 'rmsd', 'normalized_trajectory', 'movement_speed',
       'accuracy'],
      dtype='object')

In [3]:
df_char = df[['participant_id', 'task_type', 'Age', 'Cohort', 'Gestational_Age', 'mabc_percentile']]
df_char

,participant_id,task_type,Age,Cohort,Gestational_Age,mabc_percentile
0,0,0,5,Preterm,33.0,50.0
1,0,0,5,Preterm,33.0,50.0
2,0,0,5,Preterm,33.0,50.0
3,0,0,5,Preterm,33.0,50.0
4,0,0,5,Preterm,33.0,50.0
...,...,...,...,...,...,...
16315,67,1,8,Term,40.0,0.5
16316,67,1,8,Term,40.0,0.5
16317,67,1,8,Term,40.0,0.5
16318,67,1,8,Term,40.0,0.5


In [33]:
from scipy import stats
import random
term_participant_ids = df[df['Cohort'] == 'Term']['participant_id'].unique()
preterm_participant_ids = df[df['Cohort'] == 'Preterm']['participant_id'].unique()

# Randomly select 15 participants from each cohort
# If there are fewer than 15 participants in either group, take all available participants
random.seed(42)  # For reproducibility
selected_term_ids = random.sample(list(term_participant_ids), min(18, len(term_participant_ids)))
selected_preterm_ids = random.sample(list(preterm_participant_ids), min(18, len(preterm_participant_ids)))

# Filter the dataframe for these participants
term_subset = df[df['participant_id'].isin(selected_term_ids)]
preterm_subset = df[df['participant_id'].isin(selected_preterm_ids)]
print(term_subset.shape, preterm_subset.shape)

# Get the mabc_percentile values for the selected participants
term_data = term_subset['mabc_percentile'].values
preterm_data = preterm_subset['mabc_percentile'].values

# Calculate means
term_mean = np.mean(term_data)
preterm_mean = np.mean(preterm_data)

# Calculate 95% confidence intervals
def get_ci(data, confidence=0.95):
    n = len(data)
    m = np.mean(data)
    se = stats.sem(data)
    h = se * stats.t.ppf((1 + confidence) / 2, n-1)
    return m, m-h, m+h  # mean, lower bound, upper bound

term_mean, term_lower, term_upper = get_ci(term_data)
preterm_mean, preterm_lower, preterm_upper = get_ci(preterm_data)

# Perform independent t-test (using Welch's t-test for unequal variances)
t_stat, p_value = stats.ttest_ind(term_data, preterm_data, equal_var=False)

# Print results
print("T-test results comparing MABC percentile between randomly selected Term and Preterm cohorts:")
print(f"Term (n={len(term_data)}): {term_mean:.2f} [95% CI: {term_lower:.2f} to {term_upper:.2f}]")
print(f"Preterm (n={len(preterm_data)}): {preterm_mean:.2f} [95% CI: {preterm_lower:.2f} to {preterm_upper:.2f}]")
print(f"Mean difference: {term_mean - preterm_mean:.2f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

# Check which specific participant_ids were selected
print("\nSelected Term participant IDs:", selected_term_ids)
print("Selected Preterm participant IDs:", selected_preterm_ids)

(4319, 37) (4319, 37)
T-test results comparing MABC percentile between randomly selected Term and Preterm cohorts:
Term (n=4319): 39.91 [95% CI: 39.16 to 40.67]
Preterm (n=4319): 23.61 [95% CI: 22.74 to 24.49]
Mean difference: 16.30
t-statistic: 27.591
p-value: 0.0000

Selected Term participant IDs: [58, 8, 2, 19, 17, 16, 10, 7, 45, 6, 55, 37, 3, 65, 67, 15, 62, 43]
Selected Preterm participant IDs: [0, 30, 49, 22, 34, 36, 23, 53, 21, 52, 29, 47, 9, 51, 35, 39, 50, 46]


In [15]:
def format_cohort_comparison_for_paper(df):
    """
    Format a cohort comparison of participant characteristics for a research paper.
    Only uses unique participants (drops duplicates).
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing participant data
    
    Returns:
    str: Formatted text for research paper comparing cohorts
    """
    # Ensure we only use unique participants
    df_unique = df.drop_duplicates(subset=['participant_id'])
    
    # Get unique cohorts
    cohorts = df_unique['Cohort'].unique()
    
    # Calculate statistics by cohort
    results = calculate_participant_characteristics_by_cohort(df)
    
    # Format overall statistics
    text = "Participant Characteristics:\n"
    text += f"A total of {results['overall']['n_participants']} unique participants were included in the analysis. "
    
    # Add cohort distribution
    text += "Participants were distributed across cohorts as follows: "
    cohort_text = []
    for cohort in cohorts:
        cohort_count = len(df_unique[df_unique['Cohort'] == cohort])
        cohort_percent = cohort_count / len(df_unique) * 100
        cohort_text.append(f"{cohort}: {cohort_count} ({cohort_percent:.1f}%)")
    text += ", ".join(cohort_text) + ". "
    
    # Compare age across cohorts
    text += "\n\nAge characteristics by cohort:\n"
    for cohort in cohorts:
        cohort_stats = results['by_cohort'][cohort]
        text += f"- {cohort}: mean age {cohort_stats['age']['mean']:.2f} years "
        text += f"(SD = {cohort_stats['age']['std']:.2f}, "
        text += f"range = {cohort_stats['age']['min']:.1f}-{cohort_stats['age']['max']:.1f}). "
        
        # Add age group distribution for this cohort
        age_group_text = []
        for age_group, count in sorted(cohort_stats['age_group'].items()):
            percent = cohort_stats['age_group_percent'][age_group]
            age_group_text.append(f"{age_group} years: {count} ({percent:.1f}%)")
        text += f"Age groups: {', '.join(age_group_text)}.\n"
    
    # Compare task types across cohorts
    text += "\nTask type distribution by cohort:\n"
    for cohort in cohorts:
        cohort_stats = results['by_cohort'][cohort]
        task_text = []
        for task, count in cohort_stats['task_type'].items():
            percent = cohort_stats['task_type_percent'][task]
            task_text.append(f"{task}: {count} ({percent:.1f}%)")
        text += f"- {cohort}: {', '.join(task_text)}.\n"
    
    # Compare gestational age across cohorts
    text += "\nGestational age characteristics by cohort:\n"
    for cohort in cohorts:
        cohort_stats = results['by_cohort'][cohort]
        text += f"- {cohort}: mean gestational age {cohort_stats['gestational_age']['mean']:.2f} weeks "
        text += f"(SD = {cohort_stats['gestational_age']['std']:.2f}, "
        text += f"range = {cohort_stats['gestational_age']['min']:.1f}-{cohort_stats['gestational_age']['max']:.1f}).\n"
    
    # Compare MABC percentile across cohorts
    text += "\nMABC percentile characteristics by cohort:\n"
    for cohort in cohorts:
        cohort_stats = results['by_cohort'][cohort]
        text += f"- {cohort}: mean MABC percentile {cohort_stats['mabc_percentile']['mean']:.2f} "
        text += f"(SD = {cohort_stats['mabc_percentile']['std']:.2f}, "
        text += f"range = {cohort_stats['mabc_percentile']['min']:.1f}-{cohort_stats['mabc_percentile']['max']:.1f}).\n"
    
    return text

def create_cohort_comparison_table(df):
    """
    Create a summary table comparing participant characteristics across cohorts.
    Only uses unique participants (drops duplicates).
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing participant data
    
    Returns:
    pandas.DataFrame: Summary table with statistics by cohort
    """
    # Ensure we only use unique participants
    df_unique = df.drop_duplicates(subset=['participant_id'])
    
    # Get unique cohorts
    cohorts = df_unique['Cohort'].unique()
    
    # Calculate statistics for each cohort
    cohort_stats = {}
    for cohort in cohorts:
        cohort_df = df_unique[df_unique['Cohort'] == cohort]
        
        # Create age groups for this cohort
        cohort_df['age_group'] = pd.cut(cohort_df['Age'], 
                                       bins=[5, 6, 7, 8], 
                                       labels=['5-6', '6-7', '7-8'], 
                                       right=False)
        
        # Map task type
        task_map = {0: 'Unimanual', 1: 'Bimanual'}
        cohort_df['task_type_label'] = cohort_df['task_type'].map(task_map)
        
        # Store stats for this cohort
        cohort_stats[cohort] = {
            'n': len(cohort_df),
            'age_mean': cohort_df['Age'].mean(),
            'age_std': cohort_df['Age'].std(),
            'age_min': cohort_df['Age'].min(),
            'age_max': cohort_df['Age'].max(),
            'age_groups': cohort_df['age_group'].value_counts().to_dict(),
            'task_types': cohort_df['task_type_label'].value_counts().to_dict(),
            'gestational_age_mean': cohort_df['Gestational_Age'].mean(),
            'gestational_age_std': cohort_df['Gestational_Age'].std(),
            'mabc_mean': cohort_df['mabc_percentile'].mean(),
            'mabc_std': cohort_df['mabc_percentile'].std()
        }
    
    # Create comparison table
    # First, prepare the columns: Characteristic, Overall, Cohort1, Cohort2, ...
    columns = ['Characteristic', 'Overall'] + list(cohorts)
    comparison = pd.DataFrame(columns=columns)
    
    # Add rows for each characteristic
    # Number of participants
    row = ['Number of participants', len(df_unique)]
    for cohort in cohorts:
        row.append(cohort_stats[cohort]['n'])
    comparison.loc[len(comparison)] = row
    
    # Age
    row = ['Age (years), mean ± SD', 
           f"{df_unique['Age'].mean():.2f} ± {df_unique['Age'].std():.2f}"]
    for cohort in cohorts:
        row.append(f"{cohort_stats[cohort]['age_mean']:.2f} ± {cohort_stats[cohort]['age_std']:.2f}")
    comparison.loc[len(comparison)] = row
    
    # Age range
    row = ['Age range', 
           f"{df_unique['Age'].min():.1f} - {df_unique['Age'].max():.1f}"]
    for cohort in cohorts:
        row.append(f"{cohort_stats[cohort]['age_min']:.1f} - {cohort_stats[cohort]['age_max']:.1f}")
    comparison.loc[len(comparison)] = row
    
    # Age groups
    df_unique['age_group'] = pd.cut(df_unique['Age'], 
                                  bins=[5, 6, 7, 8], 
                                  labels=['5-6', '6-7', '7-8'], 
                                  right=False)
    for age_group in ['5-6', '6-7', '7-8']:
        overall_count = df_unique['age_group'].value_counts().get(age_group, 0)
        overall_percent = overall_count / len(df_unique) * 100 if len(df_unique) > 0 else 0
        
        row = [f'Age group: {age_group}', f"{overall_count} ({overall_percent:.1f}%)"]
        
        for cohort in cohorts:
            cohort_count = cohort_stats[cohort]['age_groups'].get(age_group, 0)
            cohort_percent = cohort_count / cohort_stats[cohort]['n'] * 100 if cohort_stats[cohort]['n'] > 0 else 0
            row.append(f"{cohort_count} ({cohort_percent:.1f}%)")
        
        comparison.loc[len(comparison)] = row
    
    # Task types
    task_map = {0: 'Unimanual', 1: 'Bimanual'}
    df_unique['task_type_label'] = df_unique['task_type'].map(task_map)
    for task in ['Unimanual', 'Bimanual']:
        overall_count = df_unique['task_type_label'].value_counts().get(task, 0)
        overall_percent = overall_count / len(df_unique) * 100 if len(df_unique) > 0 else 0
        
        row = [f'Task type: {task}', f"{overall_count} ({overall_percent:.1f}%)"]
        
        for cohort in cohorts:
            cohort_count = cohort_stats[cohort]['task_types'].get(task, 0)
            cohort_percent = cohort_count / cohort_stats[cohort]['n'] * 100 if cohort_stats[cohort]['n'] > 0 else 0
            row.append(f"{cohort_count} ({cohort_percent:.1f}%)")
        
        comparison.loc[len(comparison)] = row
    
    # Gestational Age
    row = ['Gestational Age (weeks), mean ± SD', 
           f"{df_unique['Gestational_Age'].mean():.2f} ± {df_unique['Gestational_Age'].std():.2f}"]
    for cohort in cohorts:
        row.append(f"{cohort_stats[cohort]['gestational_age_mean']:.2f} ± {cohort_stats[cohort]['gestational_age_std']:.2f}")
    comparison.loc[len(comparison)] = row
    
    # MABC percentile
    row = ['MABC percentile, mean ± SD', 
           f"{df_unique['mabc_percentile'].mean():.2f} ± {df_unique['mabc_percentile'].std():.2f}"]
    for cohort in cohorts:
        row.append(f"{cohort_stats[cohort]['mabc_mean']:.2f} ± {cohort_stats[cohort]['mabc_std']:.2f}")
    comparison.loc[len(comparison)] = row
    
    return comparison

def calculate_participant_characteristics_by_cohort(df):
    """
    Calculate descriptive statistics for participant characteristics grouped by cohort.
    Only uses unique participants (drops duplicates).
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing participant data with columns:
        'participant_id', 'Age', 'Cohort', 'Gestational_Age', 'mabc_percentile', 'task_type'
    
    Returns:
    dict: Dictionary containing statistics for each measure by cohort
    """
    # Ensure we only use unique participants (drop duplicates based on participant_id)
    df_unique = df.drop_duplicates(subset=['participant_id'])
    
    # Get unique cohorts
    cohorts = df_unique['Cohort'].unique()
    
    # Initialize results dictionary
    results = {
        'overall': {},
        'by_cohort': {}
    }
    
    # Overall statistics (with unique participants)
    results['overall'] = calculate_participant_characteristics(df_unique)
    
    # Calculate statistics for each cohort
    for cohort in cohorts:
        cohort_df = df_unique[df_unique['Cohort'] == cohort]
        results['by_cohort'][cohort] = calculate_participant_characteristics(cohort_df)
    
    return results


# Assuming df_char is your dataframe with the participant characteristics
# If you don't have this dataframe already, you would load it from your data source
# For example: df_char = pd.read_csv('your_data.csv')

# Example of creating a sample dataframe for testing purposes
def create_sample_dataframe(n=100):
    """
    Create a sample dataframe with participant characteristics for testing.
    
    Parameters:
    n (int): Number of participants to generate
    
    Returns:
    pandas.DataFrame: Sample dataframe with participant characteristics
    """
    np.random.seed(42)  # For reproducibility
    
    # Generate random data
    participant_ids = [f"P{i:03d}" for i in range(1, n+1)]
    ages = np.random.uniform(5.0, 8.0, n)  # Ages between 5 and 8 years
    cohorts = np.random.choice(['Control', 'Experimental'], n)
    gestational_ages = np.random.normal(39, 2, n)  # Normal distribution around 39 weeks
    mabc_percentiles = np.random.uniform(1, 99, n)  # Percentiles from 1 to 99
    task_types = np.random.choice([0, 1], n)  # 0=Unimanual, 1=Bimanual
    
    # Create dataframe
    df = pd.DataFrame({
        'participant_id': participant_ids,
        'Age': ages,
        'Cohort': cohorts,
        'Gestational_Age': gestational_ages,
        'mabc_percentile': mabc_percentiles,
        'task_type': task_types
    })
    
    return df

# Uncomment to create a sample dataframe for testing
# df_char = create_sample_dataframe(100)

def calculate_participant_characteristics(df):
    """
    Calculate descriptive statistics for participant characteristics.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing participant data with columns:
        'participant_id', 'Age', 'Cohort', 'Gestational_Age', 'mabc_percentile', 'task_type'
    
    Returns:
    dict: Dictionary containing statistics for each measure
    """
    stats = {}
    
    # Number of participants
    stats['n_participants'] = len(df)
    
    # Age statistics
    stats['age'] = {
        'mean': df['Age'].mean(),
        'std': df['Age'].std(),
        'median': df['Age'].median(),
        'min': df['Age'].min(),
        'max': df['Age'].max()
    }
    
    # Age group distribution
    # Create age groups: 5-6, 6-7, 7-8
    df['age_group'] = pd.cut(df['Age'], 
                            bins=[5, 6, 7, 8], 
                            labels=['5-6', '6-7', '7-8'], 
                            right=False)
    stats['age_group'] = df['age_group'].value_counts().to_dict()
    stats['age_group_percent'] = (df['age_group'].value_counts(normalize=True) * 100).to_dict()
    
    # Cohort distribution
    stats['cohort'] = df['Cohort'].value_counts().to_dict()
    stats['cohort_percent'] = (df['Cohort'].value_counts(normalize=True) * 100).to_dict()
    
    # Task type distribution (0=Unimanual, 1=Bimanual)
    task_map = {0: 'Unimanual', 1: 'Bimanual'}
    df['task_type_label'] = df['task_type'].map(task_map)
    stats['task_type'] = df['task_type_label'].value_counts().to_dict()
    stats['task_type_percent'] = (df['task_type_label'].value_counts(normalize=True) * 100).to_dict()
    
    # Gestational Age statistics
    stats['gestational_age'] = {
        'mean': df['Gestational_Age'].mean(),
        'std': df['Gestational_Age'].std(),
        'median': df['Gestational_Age'].median(),
        'min': df['Gestational_Age'].min(),
        'max': df['Gestational_Age'].max()
    }
    
    # MABC percentile statistics
    stats['mabc_percentile'] = {
        'mean': df['mabc_percentile'].mean(),
        'std': df['mabc_percentile'].std(),
        'median': df['mabc_percentile'].median(),
        'min': df['mabc_percentile'].min(),
        'max': df['mabc_percentile'].max()
    }
    
    return stats

def create_summary_table(df):
    """
    Create a summary table for participant characteristics.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing participant data
    
    Returns:
    pandas.DataFrame: Summary table with statistics
    """
    # Create a summary dataframe
    summary = pd.DataFrame(columns=['Characteristic', 'Value'])
    
    # Add number of participants
    summary.loc[len(summary)] = ['Number of participants', len(df)]
    
    # Add age statistics
    summary.loc[len(summary)] = ['Age (years), mean ± SD', f"{df['Age'].mean():.2f} ± {df['Age'].std():.2f}"]
    summary.loc[len(summary)] = ['Age range', f"{df['Age'].min():.1f} - {df['Age'].max():.1f}"]
    
    # Add age group distribution
    df['age_group'] = pd.cut(df['Age'], 
                           bins=[5, 6, 7, 8], 
                           labels=['5-6', '6-7', '7-8'], 
                           right=False)
    for age_group, count in df['age_group'].value_counts().sort_index().items():
        percent = count / len(df) * 100
        summary.loc[len(summary)] = [f'Age group: {age_group}', f"{count} ({percent:.1f}%)"]
    
    # Add cohort distribution
    for cohort, count in df['Cohort'].value_counts().items():
        percent = count / len(df) * 100
        summary.loc[len(summary)] = [f'Cohort: {cohort}', f"{count} ({percent:.1f}%)"]
    
    # Add task type distribution
    task_map = {0: 'Unimanual', 1: 'Bimanual'}
    df['task_type_label'] = df['task_type'].map(task_map)
    for task, count in df['task_type_label'].value_counts().items():
        percent = count / len(df) * 100
        summary.loc[len(summary)] = [f'Task type: {task}', f"{count} ({percent:.1f}%)"]
    
    # Add gestational age statistics
    summary.loc[len(summary)] = ['Gestational Age (weeks), mean ± SD', 
                                f"{df['Gestational_Age'].mean():.2f} ± {df['Gestational_Age'].std():.2f}"]
    summary.loc[len(summary)] = ['Gestational Age range', 
                               f"{df['Gestational_Age'].min():.1f} - {df['Gestational_Age'].max():.1f}"]
    
    # Add MABC percentile statistics
    summary.loc[len(summary)] = ['MABC percentile, mean ± SD', 
                               f"{df['mabc_percentile'].mean():.2f} ± {df['mabc_percentile'].std():.2f}"]
    summary.loc[len(summary)] = ['MABC percentile range', 
                              f"{df['mabc_percentile'].min():.1f} - {df['mabc_percentile'].max():.1f}"]
    
    return summary

# To use the functions:
stats = calculate_participant_characteristics(df_char)
summary_table = create_summary_table(df_char)
print(summary_table)

# Example of how to format the statistics for a research paper
def format_for_research_paper(df):
    """
    Format participant characteristics statistics for a research paper.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing participant data
    
    Returns:
    str: Formatted text for research paper
    """
    stats = calculate_participant_characteristics(df)
    
    text = "Participant Characteristics:\n"
    text += f"A total of {stats['n_participants']} participants were included in the analysis. "
    text += f"The mean age was {stats['age']['mean']:.2f} years (SD = {stats['age']['std']:.2f}, "
    text += f"range = {stats['age']['min']:.1f}-{stats['age']['max']:.1f}). "
    
    # Format age group information
    age_group_text = []
    for age_group, count in sorted(stats['age_group'].items()):
        percent = stats['age_group_percent'][age_group]
        age_group_text.append(f"{age_group} years: {count} ({percent:.1f}%)")
    
    text += f"Participants were distributed across age groups as follows: {', '.join(age_group_text)}. "
    
    # Format cohort information
    cohort_text = []
    for cohort, count in stats['cohort'].items():
        percent = stats['cohort_percent'][cohort]
        cohort_text.append(f"{cohort}: {count} ({percent:.1f}%)")
    
    text += f"Participants were distributed across cohorts as follows: {', '.join(cohort_text)}. "
    
    # Format task type information
    task_text = []
    for task, count in stats['task_type'].items():
        percent = stats['task_type_percent'][task]
        task_text.append(f"{task}: {count} ({percent:.1f}%)")
    
    text += f"Tasks were distributed as follows: {', '.join(task_text)}. "
    
    text += f"The mean gestational age was {stats['gestational_age']['mean']:.2f} weeks "
    text += f"(SD = {stats['gestational_age']['std']:.2f}, "
    text += f"range = {stats['gestational_age']['min']:.1f}-{stats['gestational_age']['max']:.1f}). "
    
    text += f"The mean MABC percentile was {stats['mabc_percentile']['mean']:.2f} "
    text += f"(SD = {stats['mabc_percentile']['std']:.2f}, "
    text += f"range = {stats['mabc_percentile']['min']:.1f}-{stats['mabc_percentile']['max']:.1f})."
    
    return text

# Example usage with the cohort comparison functions:
# By cohort statistics
stats_by_cohort = calculate_participant_characteristics_by_cohort(df_char)
print(stats_by_cohort)

# Cohort comparison table
comparison_table = create_cohort_comparison_table(df_char)
print(comparison_table)

# Cohort comparison for research paper
cohort_paper_text = format_cohort_comparison_for_paper(df_char)
print(cohort_paper_text)

# Example of how to use the functions with sample data:
# Sample usage demonstration
# if __name__ == "__main__":
    # Create sample data
df_sample = create_sample_dataframe(100)

# Add some duplicate participant IDs to demonstrate deduplication
duplicates = df_sample.sample(20).copy()
duplicates['task_type'] = 1 - duplicates['task_type']  # Flip task type for duplicates
df_with_duplicates = pd.concat([df_sample, duplicates])

print(f"Total rows: {len(df_with_duplicates)}")
print(f"Unique participants: {df_with_duplicates['participant_id'].nunique()}")

# Calculate statistics with unique rows only
stats_by_cohort = calculate_participant_characteristics_by_cohort(df_with_duplicates)

# Create comparison table
comparison_table = create_cohort_comparison_table(df_with_duplicates)

# Format for research paper
cohort_paper_text = format_cohort_comparison_for_paper(df_with_duplicates)

print("\nComparison Table:")
print(comparison_table)

print("\nFormatted for Research Paper:")
print(cohort_paper_text)

/var/folders/c5/gxvjzbzd36x1q68jgw7v4zl80000gn/T/ipykernel_84879/2907653457.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['age_group'] = pd.cut(df['Age'],
/var/folders/c5/gxvjzbzd36x1q68jgw7v4zl80000gn/T/ipykernel_84879/2907653457.py:315: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['task_type_label'] = df['task_type'].map(task_map)
/var/folders/c5/gxvjzbzd36x1q68jgw7v4zl80000gn/T/ipykernel_84879/2907653457.py:360: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from

                        Characteristic          Value
0               Number of participants          16320
1               Age (years), mean ± SD    6.69 ± 1.06
2                            Age range      5.0 - 8.0
3                       Age group: 5-6   3120 (19.1%)
4                       Age group: 6-7   3120 (19.1%)
5                       Age group: 7-8   5760 (35.3%)
6                         Cohort: Term  12000 (73.5%)
7                      Cohort: Preterm   4320 (26.5%)
8                 Task type: Unimanual   8160 (50.0%)
9                  Task type: Bimanual   8160 (50.0%)
10  Gestational Age (weeks), mean ± SD   37.06 ± 3.93
11               Gestational Age range    23.0 - 42.0
12          MABC percentile, mean ± SD  39.10 ± 30.63
13               MABC percentile range     0.5 - 95.0
{'overall': {'n_participants': 68, 'age': {'mean': 6.6911764705882355, 'std': 1.068648929984729, 'median': 7.0, 'min': 5, 'max': 8}, 'age_group': {'7-8': 24, '5-6': 13, '6-7': 13}, 'age_grou

/var/folders/c5/gxvjzbzd36x1q68jgw7v4zl80000gn/T/ipykernel_84879/2907653457.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['age_group'] = pd.cut(df['Age'],
/var/folders/c5/gxvjzbzd36x1q68jgw7v4zl80000gn/T/ipykernel_84879/2907653457.py:315: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['task_type_label'] = df['task_type'].map(task_map)
/var/folders/c5/gxvjzbzd36x1q68jgw7v4zl80000gn/T/ipykernel_84879/2907653457.py:302: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from


Comparison Table:
                       Characteristic        Overall        Control  \
0              Number of participants            100             51   
1              Age (years), mean ± SD    6.41 ± 0.89    6.43 ± 0.87   
2                           Age range      5.0 - 8.0      5.1 - 8.0   
3                      Age group: 5-6     41 (41.0%)     20 (39.2%)   
4                      Age group: 6-7     28 (28.0%)     14 (27.5%)   
5                      Age group: 7-8     31 (31.0%)     17 (33.3%)   
6                Task type: Unimanual     60 (60.0%)     31 (60.8%)   
7                 Task type: Bimanual     40 (40.0%)     20 (39.2%)   
8  Gestational Age (weeks), mean ± SD   39.23 ± 1.97   38.95 ± 1.81   
9          MABC percentile, mean ± SD  44.53 ± 28.35  39.02 ± 30.22   

    Experimental  
0             49  
1    6.39 ± 0.92  
2      5.0 - 7.9  
3     21 (42.9%)  
4     14 (28.6%)  
5     14 (28.6%)  
6     29 (59.2%)  
7     20 (40.8%)  
8   39.53 ± 2.10  
9  50.25 

In [9]:
df_char

,participant_id,Age,Cohort,Gestational_Age,mabc_percentile
0,0,5,Preterm,33.0,50.0
1,0,5,Preterm,33.0,50.0
2,0,5,Preterm,33.0,50.0
3,0,5,Preterm,33.0,50.0
4,0,5,Preterm,33.0,50.0
...,...,...,...,...,...
16315,67,8,Term,40.0,0.5
16316,67,8,Term,40.0,0.5
16317,67,8,Term,40.0,0.5
16318,67,8,Term,40.0,0.5


In [17]:
def age_group_distribution(df):
    """
    A simple standalone function that calculates the count and percentage 
    of participants in each age group (5-6, 6-7, 7-8), grouped by cohort.
    Only counts unique participants.
    
    Parameters:
    df (pandas.DataFrame): DataFrame with 'participant_id', 'Age', and 'Cohort' columns
    
    Returns:
    pandas.DataFrame: Formatted table showing age group distribution by cohort
    """
    # Ensure we only use unique participants
    df_unique = df.drop_duplicates(subset=['participant_id'])
    
    # Create age groups
    df_unique['age_group'] = pd.cut(df_unique['Age'], 
                                  bins=[0, 6, 7, 10], 
                                  labels=['5-6', '6-7', '7-8'], 
                                  right=False)
    
    # Count participants in each age group by cohort
    counts = pd.crosstab(df_unique['age_group'], df_unique['Cohort'])
    
    # Calculate percentages within each cohort
    percentages = counts.copy()
    for col in percentages.columns:
        percentages[col] = (percentages[col] / percentages[col].sum() * 100).round(1)
    
    # Create the formatted result
    result = pd.DataFrame(index=counts.index)
    
    # Format as count (percentage%)
    for cohort in counts.columns:
        result[cohort] = [f"{counts.loc[age, cohort]} ({percentages.loc[age, cohort]}%)" 
                      for age in counts.index]
    
    # Add an Overall column
    overall_counts = df_unique['age_group'].value_counts().sort_index()
    overall_percentages = (overall_counts / overall_counts.sum() * 100).round(1)
    
    result['Overall'] = [f"{overall_counts[age]} ({overall_percentages[age]}%)" 
                      for age in counts.index]
    
    # Add totals row
    totals = {}
    for col in result.columns:
        if col == 'Overall':
            count = df_unique.shape[0]
        else:
            count = df_unique[df_unique['Cohort'] == col].shape[0]
        totals[col] = f"{count} (100.0%)"
    
    result.loc['Total'] = totals
    
    return result

# Example usage:
age_distribution = age_group_distribution(df_char)
print(age_distribution)

               Preterm         Term      Overall
age_group                                       
5-6          6 (33.3%)    7 (14.0%)   13 (19.1%)
6-7           1 (5.6%)   12 (24.0%)   13 (19.1%)
7-8         11 (61.1%)   31 (62.0%)   42 (61.8%)
Total      18 (100.0%)  50 (100.0%)  68 (100.0%)


/var/folders/c5/gxvjzbzd36x1q68jgw7v4zl80000gn/T/ipykernel_84879/3227765276.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique['age_group'] = pd.cut(df_unique['Age'],
